# Axis 2: Unified Three-System Probing — Resilient Edition

Probes verb-spatial binding in **three** generative systems on AGD20K (36 affordance categories):

1. **Flux** (text-to-image) — known-good baseline replicating Zhang et al. (H2a)
2. **Cosmos-Predict2-2B-Video2World** — base video diffusion (H2b)
3. **Cosmos-Policy-ALOHA-Predict2-2B** — manipulation-fine-tuned variant (H2c)

## Required
- **Colab Pro+ recommended** (24h sessions). Plain Pro works but you'll need 2-3 sessions.
- **A100 40GB GPU** — Cosmos needs ~32 GB.
- **HuggingFace account** with access to FLUX.1-dev (gated) and Cosmos models.
- **AGD20K dataset** at `/content/drive/MyDrive/datasets/AGD20K` (or let cell 2 try direct download).

## Time budget (pilot, 30 samples × 36 categories per system)
| Phase | Time |
|---|---|
| Cell 0–2: install + auth + dataset | ~10 min one-time |
| Cell 3 Flux pilot (schnell, 4 steps) | ~45 min |
| Cell 4 Cosmos V2W: smoke + pilot | ~3 hrs |
| Cell 5 Cosmos Policy: smoke + pilot | ~3 hrs |
| Cell 6 three-way comparison | ~5 min |
| **TOTAL pilot** | **~7 hrs** |

## Resilience features
- **Resume:** every probing script writes CSV incrementally with `fsync` after every row, and `--resume` (default on) skips done sample IDs.
- **Persistent HF cache** on Drive — models survive disconnects (cell 1).
- **Periodic git push** of result CSVs (--commit_every flag) — progress lives on GitHub.
- **`./results/` symlinked to Drive** — artifacts persist mid-run (cell 2).
- **Explicit VRAM cleanup** between systems (cells 3.5, 4.5).
- **Optional JS keepalive** (cell 7) — suppresses 90-min idle disconnect.
- **Fail-fast smoke tests** (stages 4a, 5a) — extractor bugs surface in 3 min, not 3 hours.

## If your session dies mid-run
Open the notebook fresh, run cells 0–2 (idempotent, ~2 min). Re-run whichever cell was running; the script reports `Resume: N samples already done` and continues.

## Cell 0 — Repo + dependencies + git auth

Set `GH_PAT` in Colab Secrets (`🔑` icon, left sidebar) for git pushes.

In [ ]:
import os, subprocess
from pathlib import Path

REPO = 'aleksantari/VLA-affordance'
BRANCH = 'nj-features'
REPO_DIR = '/content/VLA-affordance'

GH_PAT = ''
try:
    from google.colab import userdata
    GH_PAT = userdata.get('GH_PAT') or ''
    print('✓ GH_PAT found' if GH_PAT else '⚠ No GH_PAT — push disabled, results on Drive only')
except Exception:
    print('⚠ Colab userdata unavailable — push disabled')

remote = f'https://{GH_PAT}@github.com/{REPO}.git' if GH_PAT else f'https://github.com/{REPO}.git'
if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', remote, REPO_DIR], check=True)
%cd /content/VLA-affordance
subprocess.run(['git', 'remote', 'set-url', 'origin', remote], check=True)
subprocess.run(['git', 'fetch', 'origin'], check=True)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['git', 'pull', 'origin', BRANCH], check=True)
subprocess.run(['git', 'config', 'user.email', 'jajda.n@northeastern.edu'], check=True)
subprocess.run(['git', 'config', 'user.name', 'nitik1998 (Colab)'], check=True)

!pip install -e . -q
!pip install -r requirements_axis2.txt -q
print('\n✓ Setup complete')

## Cell 1 — GPU check + persistent HF cache on Drive

Redirects `~/.cache/huggingface` to Drive so the 24+ GB of model weights survive runtime disconnects.

In [ ]:
import torch, os
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime > Change runtime type > A100')
name = torch.cuda.get_device_name()
vram = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f'GPU: {name}  VRAM: {vram:.1f} GB')
if vram < 35:
    print('⚠ Cosmos V2W needs ~32 GB; this GPU may OOM. Use --cpu_offload as fallback.')

from google.colab import drive
drive.mount('/content/drive')

DRIVE_HF_CACHE = '/content/drive/MyDrive/hf_cache'
os.makedirs(DRIVE_HF_CACHE, exist_ok=True)
os.environ['HF_HOME'] = DRIVE_HF_CACHE
os.environ['HUGGINGFACE_HUB_CACHE'] = DRIVE_HF_CACHE + '/hub'
os.environ['TRANSFORMERS_CACHE'] = DRIVE_HF_CACHE + '/transformers'
print(f'✓ HF_HOME = {DRIVE_HF_CACHE}')

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    if HF_TOKEN:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
        print('✓ HF login OK')
    else:
        print('⚠ No HF_TOKEN. Set one at hf.co/settings/tokens; add as Colab secret HF_TOKEN.')
except Exception as e:
    print(f'⚠ HF login skipped: {e}')

## Cell 2 — Persistent results dir + AGD20K

Results symlinked to Drive (survives disconnects). AGD20K symlinked from Drive if pre-staged; falls back to direct download.

In [ ]:
import os, shutil
from pathlib import Path

DRIVE_RESULTS = Path('/content/drive/MyDrive/VLA-affordance-results')
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
if not Path('./results').exists() or not Path('./results').is_symlink():
    if Path('./results').exists():
        shutil.rmtree('./results')
    os.symlink(str(DRIVE_RESULTS), './results')
print(f'✓ ./results -> {DRIVE_RESULTS}')

# AGD20K — three locations are checked, in order:
#   1. /content/drive/MyDrive/datasets/AGD20K  (preferred, persistent)
#   2. /content/drive/MyDrive/LBV Project/AGD20K.zip  (one-time unzip)
#   3. embedded LOCATE Drive ID (last-resort direct download)
DRIVE_AGD20K_DIR = Path('/content/drive/MyDrive/datasets/AGD20K')
DRIVE_AGD20K_ZIP = Path('/content/drive/MyDrive/LBV Project/AGD20K.zip')

if DRIVE_AGD20K_DIR.exists() and any(DRIVE_AGD20K_DIR.iterdir()):
    print(f'✓ Found unzipped AGD20K at {DRIVE_AGD20K_DIR}')
    !python data/download_agd20k.py --from_drive "{DRIVE_AGD20K_DIR}" --output_dir ./data/agd20k
elif DRIVE_AGD20K_ZIP.exists():
    print(f'✓ Found zip at {DRIVE_AGD20K_ZIP} — unzipping (one-time)')
    !python data/download_agd20k.py --from_zip "{DRIVE_AGD20K_ZIP}" --output_dir ./data/agd20k
    # Persist the unzipped copy to Drive at the canonical path so future
    # sessions skip this step (saves ~30 sec per session and ~2 GB writes)
    print(f'Copying unzipped AGD20K → {DRIVE_AGD20K_DIR} for future sessions...')
    DRIVE_AGD20K_DIR.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree('./data/agd20k', str(DRIVE_AGD20K_DIR), dirs_exist_ok=True)
    print(f'✓ Persisted to {DRIVE_AGD20K_DIR}')
else:
    print(f'⚠ No Drive copy found. Falling back to direct download (LOCATE mirror).')
    !python data/download_agd20k.py --output_dir ./data/agd20k

!python data/download_agd20k.py --validate_only --output_dir ./data/agd20k

## Cell 3 — System A: Flux (H2a)

`--commit_every 100` pushes the per-sample CSV every 100 samples. `--resume` (default on) skips samples already in the CSV.

In [ ]:
!python scripts/09_setup_flux.py --model schnell

In [ ]:
# Pilot — 30 samples per category, schnell. Resumes from CSV if interrupted.
!python scripts/10_run_interaction_probing.py \
    --model schnell \
    --max_per_category 30 \
    --commit_every 100 \
    --save_attention_maps

In [ ]:
# Final-quality (uncomment when pilot looks good):
# !python scripts/10_run_interaction_probing.py --model dev --commit_every 100 --save_attention_maps

## Cell 3.5 — Free Flux from VRAM

Critical: without this, loading Cosmos OOMs.

In [ ]:
import gc, torch
for v in list(globals()):
    if v.startswith('_') or v in ('gc', 'torch'):
        continue
    try:
        obj = globals()[v]
        if 'Pipeline' in type(obj).__name__ or 'Flux' in type(obj).__name__:
            del globals()[v]
    except Exception:
        pass
gc.collect()
torch.cuda.empty_cache()
print(f'VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated')

## Cell 4 — System B: Cosmos-Predict2 V2W (H2b)

Three stages, fail-fast ordered:
1. **4a — Smoke test (~3 min):** loads model, runs one inference on a synthetic image. Validates extractor mechanics.
2. **4b — Mini-pilot (~5 min):** 3 categories × 3 AGD20K samples. Validates dataset integration.
3. **4c — Full pilot (~3 hrs):** 30 samples × 36 categories. Resumes incrementally.

If 4a fails: do NOT run 4c. Fix the bug, re-push to nj-features from local, re-pull on Colab.

In [ ]:
# Stage 4a — fail-fast smoke test
!python scripts/09b_setup_cosmos.py --system cosmos_predict2_v2w \
    --num_inference_steps 4 --num_frames 5 --height 320 --width 320

In [ ]:
# Stage 4b — mini-pilot to validate AGD20K integration
!python scripts/10b_run_cosmos_probing.py \
    --system cosmos_predict2_v2w \
    --categories cut hold pour \
    --max_per_category 3 \
    --num_inference_steps 8

In [ ]:
# Stage 4c — full pilot. Resumes if interrupted.
!python scripts/10b_run_cosmos_probing.py \
    --system cosmos_predict2_v2w \
    --max_per_category 30 \
    --num_inference_steps 12 \
    --commit_every 50

## Cell 4.5 — Free Cosmos V2W from VRAM

In [ ]:
import gc, torch
for v in list(globals()):
    if v.startswith('_') or v in ('gc', 'torch'):
        continue
    try:
        obj = globals()[v]
        if 'Pipeline' in type(obj).__name__ or 'Cosmos' in type(obj).__name__:
            del globals()[v]
    except Exception:
        pass
gc.collect()
torch.cuda.empty_cache()
print(f'VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated')

## Cell 5 — System C: Cosmos Policy (H2c)

Same three-stage fail-fast pattern. Same architecture as Cosmos V2W; manipulation-fine-tuned weights.

In [ ]:
# Stage 5a — fail-fast smoke test
!python scripts/09b_setup_cosmos.py --system cosmos_policy \
    --num_inference_steps 4 --num_frames 5 --height 320 --width 320

In [ ]:
# Stage 5b — mini-pilot
!python scripts/10b_run_cosmos_probing.py \
    --system cosmos_policy \
    --categories cut hold pour \
    --max_per_category 3 \
    --num_inference_steps 8

In [ ]:
# Stage 5c — full pilot. Resumes if interrupted.
!python scripts/10b_run_cosmos_probing.py \
    --system cosmos_policy \
    --max_per_category 30 \
    --num_inference_steps 12 \
    --commit_every 50

## Cell 6 — Three-way comparison + final push

In [ ]:
!python scripts/12_three_way_comparison.py

In [ ]:
from IPython.display import Image, display
from pathlib import Path
for f in sorted(Path('./results/figures/axis2').glob('three_way_*.png')):
    print(f.name); display(Image(str(f), width=900))
import json, pprint
with open('./results/tables/axis2_hypothesis_tests.json') as fp:
    pprint.pp(json.load(fp))

In [ ]:
!git add results/tables results/figures/axis2
!git status
!git commit -m 'research(results): Axis 2 pilot — three-way comparison artifacts' || echo 'nothing new to commit'
!git push origin nj-features || echo '⚠ push failed; results are still on Drive at /content/drive/MyDrive/VLA-affordance-results'

## Cell 7 — (Optional) Idle keepalive

Fires a JS heartbeat every 30s to suppress Colab's idle disconnect (~90 min). Doesn't extend the absolute session cap (12h Pro / 24h Pro+).

In [ ]:
from IPython.display import display, Javascript
display(Javascript('''
function ColabKeepalive() {
  console.log('keepalive ping');
  document.querySelector('colab-toolbar-button#connect')?.click();
}
setInterval(ColabKeepalive, 30000);
'''))
print('✓ Keepalive armed (every 30s)')